# 6주차 예제 — Customer Personality Analysis 군집분석 (Clustering)

앞선 주차에서는 정답(target)이 있는 지도학습인 회귀와 분류를 살펴보았습니다. 이번 주에는 정답이 주어지지 않은 상태에서 **비슷한 고객을 하나의 그룹으로 묶는** 비지도학습 방법인 K-means 군집분석을 배웁니다.

1. K-means란 무엇인가 (간단한 예제로 원리부터)
2. 데이터 불러오기 & 결측치 처리
3. 파생변수 만들기
4. 이상치 후보 확인 및 처리
5. 스케일링
6. 적정 K 찾기 — Elbow Method
7. 적정 K 찾기 — Silhouette Score
8. 최종 K-means 학습
9. 클러스터 특성 해석


## Part 1. K-means란 무엇인가 (간단한 예제)

**K-means**는 가까운 데이터끼리 묶는 과정을 반복하여 군집을 찾는 알고리즘입니다. 다음 네 단계를 차례로 살펴보겠습니다.

1. K개의 초기 중심점(centroid)을 정합니다.
2. 각 데이터를 **가장 가까운 중심점**의 그룹에 배정합니다.
3. 각 그룹의 **평균 위치로 중심점을 이동합니다.**
4. 중심점의 위치가 더 이상 변하지 않을 때까지 2~3단계를 반복합니다.

먼저 숫자 6개인 `[1, 2, 3, 20, 21, 22]`를 이용해 계산 과정을 살펴보겠습니다. 두 중심점은 각각 1과 20에서 시작합니다.


In [ ]:
import numpy as np

points = np.array([1, 2, 3, 20, 21, 22])
c1, c2 = 1, 20  # 중심점 초기 위치

for it in range(5):
    dist_to_c1 = np.abs(points - c1)
    dist_to_c2 = np.abs(points - c2)
    group1 = points[dist_to_c1 <= dist_to_c2]
    group2 = points[dist_to_c1 > dist_to_c2]
    new_c1, new_c2 = group1.mean(), group2.mean()
    print(f'{it}번째 반복: c1={c1:.1f}->{new_c1:.1f} (그룹={group1}),  c2={c2:.1f}->{new_c2:.1f} (그룹={group2})')
    if new_c1 == c1 and new_c2 == c2:
        print('중심점이 더 안 움직임 -> 종료')
        break
    c1, c2 = new_c1, new_c2


두 번 반복하면 중심점이 2.0과 21.0으로 수렴합니다. 이어서 scikit-learn의 `KMeans`도 같은 결과를 내는지 확인해 보겠습니다.


In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(points.reshape(-1, 1))
print('중심점:', km.cluster_centers_.ravel())
print('그룹 배정:', km.labels_)


손으로 계산한 중심점인 2.0과 21.0이 동일하게 구해졌습니다. **K-means의 핵심은 '그룹 배정 → 그룹 평균으로 중심점 이동'을 반복하는 과정입니다.** 이때 가까운 정도를 거리로 계산하므로, 변수마다 단위가 다르면 특정 변수가 결과에 더 크게 반영될 수 있습니다. 다음 예제에서 그 영향을 살펴보겠습니다.


### 스케일링이 필요한 이유

6명의 소득과 만족도를 이용해 두 그룹을 나누어 보겠습니다. 이 예제에서 소득은 모두 3만 원대로 비슷하며 군집을 구분하는 기준으로 사용하지 않도록 구성했습니다. 반면 만족도는 2~3점과 8~9점의 두 그룹으로 구분됩니다.


In [ ]:
import pandas as pd

toy2 = pd.DataFrame({
    '소득': [30000, 30500, 31000, 30200, 30800, 30100],  # 그룹과 무관한 잡음 수준의 차이
    '만족도': [2, 2, 3, 8, 9, 8],                          # 진짜 의미 있는 차이
})
toy2


In [ ]:
km_raw = KMeans(n_clusters=2, random_state=42, n_init=10).fit(toy2)
print('스케일링 없이 그룹 배정:', km_raw.labels_)


만족도만 살펴보면 앞의 세 명과 뒤의 세 명이 서로 다른 그룹으로 구분될 것으로 예상할 수 있지만, 실제 결과는 다르게 나타납니다. 만족도의 차이는 6~7점인 반면 소득의 차이는 수백에서 천 단위이므로, 단위가 큰 소득이 거리 계산에 더 크게 반영되었기 때문입니다. 이 예제에서는 소득 차이를 군집 구분 기준으로 의도하지 않았으므로, `StandardScaler`로 두 변수의 기준을 맞춘 뒤 결과가 어떻게 달라지는지 확인해 보겠습니다.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(toy2)
km_scaled = KMeans(n_clusters=2, random_state=42, n_init=10).fit(scaled)
print('스케일링 후 그룹 배정:', km_scaled.labels_)


스케일링 후에는 앞의 세 명과 뒤의 세 명이 만족도에 따라 구분됩니다. 이 결과를 통해 변수의 단위를 맞추면 각 변수의 정보가 거리 계산에 보다 균형 있게 반영된다는 점을 확인할 수 있습니다. **따라서 단위가 다른 변수를 사용하는 K-means에서는 스케일링이 중요한 전처리 단계입니다.** Part 5에서 실제 데이터에도 같은 과정을 적용하겠습니다.


## Part 2. 데이터 불러오기와 결측치 처리

이번 실습에서는 2,240명의 마케팅 캠페인 고객 데이터를 사용합니다. 데이터는 아래 Kaggle 페이지에서 내려받을 수 있습니다.

- 데이터 다운로드: [Customer Personality Analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis)
- 파일 형식: 탭(`\t`)으로 구분된 `marketing_campaign.csv`
- 권장 경로: `../dataset/extracted/Customer Personality Analysis/marketing_campaign.csv`

압축을 해제한 파일을 권장 경로에 배치하면 아래 코드를 그대로 실행할 수 있습니다.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../dataset/extracted/Customer Personality Analysis/marketing_campaign.csv', sep='\t')
df.shape


In [ ]:
df.isna().sum()[df.isna().sum() > 0]


In [ ]:
# Income 결측은 전체의 1%뿐이라 채우기보다 제거가 더 단순하고 안전합니다
df = df.dropna(subset=['Income']).reset_index(drop=True)
df.shape


## Part 3. 파생변수 만들기

고객의 특성을 한눈에 비교할 수 있도록 나이, 총 지출액, 구매 활동 합계, 동거 자녀 수를 파생변수로 만듭니다.


In [ ]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
ref_year = df['Dt_Customer'].dt.year.max()
df['Age'] = ref_year - df['Year_Birth']

mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['Total_Spending'] = df[mnt_cols].sum(axis=1)

purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
df['Purchase_Activity_Sum'] = df[purchase_cols].sum(axis=1)

df['Children_at_Home'] = df['Kidhome'] + df['Teenhome']

df[['Age', 'Total_Spending', 'Purchase_Activity_Sum', 'Children_at_Home']].describe()


## Part 4. 이상치 후보 확인 및 처리

군집분석은 **거리**를 기준으로 데이터를 묶기 때문에 이상치의 영향을 크게 받을 수 있습니다. 극단적으로 큰 값이 있으면 중심점과 데이터 사이의 거리가 달라질 수 있으므로, 분포와 도메인 기준을 함께 확인하는 과정이 필요합니다.


In [ ]:
print('Age > 100인 사람 수:', (df['Age'] > 100).sum())
print('Income 상위 5개:')
print(df['Income'].sort_values(ascending=False).head())


`Year_Birth`가 1900년 근처인 100세 초과 관측치 3개와 소득이 66만 6666인 관측치 1개가 확인됩니다. 이 값들은 데이터 입력 오류일 가능성과 실제 관측값일 가능성을 함께 검토해야 하는 이상치 후보입니다. 이 실습에서는 일반적인 고객군을 분석한다는 목적과 변수의 분포를 고려하여 군집분석 전에 제외합니다.


In [ ]:
df = df[(df['Age'] <= 100) & (df['Income'] < 200000)].reset_index(drop=True)
df.shape


## Part 5. 스케일링

Part 1의 소득과 만족도 예제에서 살펴본 것과 같은 이유로 스케일링을 적용합니다. `Income`은 수만 단위이고 `Children_at_Home`은 0~3 범위이므로, 원래 값을 그대로 사용하면 `Income`의 차이가 거리 계산에 지나치게 크게 반영될 수 있습니다.


In [ ]:
features = ['Income', 'Age', 'Total_Spending', 'Recency', 'Children_at_Home', 'Purchase_Activity_Sum']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=features).describe().loc[['mean', 'std']]


## Part 6. 적정 K 찾기 — Elbow Method

K를 늘리면 각 군집 내부의 거리 제곱합인 inertia는 항상 감소합니다. Elbow Method에서는 K를 하나 더 늘렸을 때 inertia의 개선 폭이 눈에 띄게 작아지는 지점을 그래프의 팔꿈치 모양으로 찾습니다.


In [ ]:
K_range = range(2, 9)
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, marker='o')
ax.set_xlabel('K')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method')
plt.show()


이 그래프에서는 곡선이 완만하게 이어져 팔꿈치 지점을 하나로 정하기가 어렵습니다. 실제 데이터에서도 이처럼 경계가 분명하지 않을 수 있습니다. 이럴 때는 Elbow 결과와 다음 절의 Silhouette Score를 함께 비교하면 K를 선택하는 근거를 보완할 수 있습니다.


## Part 7. 적정 K 찾기 — Silhouette Score

Silhouette Score는 각 데이터가 자신의 군집에는 가깝고 다른 군집에는 먼 정도를 -1부터 1까지의 값으로 나타냅니다. 값이 높을수록 군집 내부의 응집도와 군집 간 분리도가 함께 좋다고 해석할 수 있습니다.


In [ ]:
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), sil_scores, marker='o', color='orange')
ax.set_xlabel('K')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score by K')
plt.show()


**Silhouette Score만 비교하면 K=2일 때의 값이 가장 높습니다.** 그러나 K=2의 군집 특성을 살펴보면 고객을 주로 소득 수준에 따라 나누어, 마케팅에 필요한 세부 고객 유형을 충분히 보여주지 못할 수 있습니다. 따라서 하나의 지표만으로 K를 정하기보다 군집의 해석 가능성과 활용 목적을 함께 고려하는 것이 좋습니다. 이 실습에서는 이러한 기준을 바탕으로 **K=4**를 선택합니다.


## Part 8. 최종 K-means 학습


In [ ]:
k_final = 4
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10).fit(X_scaled)
df['Cluster'] = km_final.labels_
df['Cluster'].value_counts().sort_index()


## Part 9. 클러스터 특성 해석

클러스터별 변수의 평균을 비교하여 각 군집의 특징을 이해하기 쉬운 표현으로 정리합니다.


In [ ]:
cluster_summary = df.groupby('Cluster')[features].mean()
cluster_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.groupby('Cluster')['Income'].mean().plot(kind='bar', ax=axes[0], title='평균 Income')
df.groupby('Cluster')['Total_Spending'].mean().plot(kind='bar', ax=axes[1], title='평균 Total_Spending')
df.groupby('Cluster')['Children_at_Home'].mean().plot(kind='bar', ax=axes[2], title='평균 동거 자녀 수')
plt.tight_layout()
plt.show()


**군집 해석 (예시)**

| Cluster | 특징 | 이름 |
|---|---|---|
| 3 | 소득 최고(77,767), 지출 최고(1,432), 자녀 거의 없음(0.10) | **VIP 고객** |
| 2 | 중상위 소득(60,523), 지출 많고(793) 구매 활동 합계도 큼(21.6) | **우량 고객** |
| 0 | 중하위 소득(41,486), 지출 적음(125), 동거 자녀 많음(1.92) | **자녀 동거 알뜰 고객** |
| 1 | 소득 최저(30,703), 나이 최연소(37.2), 지출 최저(110) | **젊은 저관여 고객** |

이 데이터에서는 **Income과 Total_Spending이 비슷한 방향으로 나타나고, Children_at_Home은 반대 방향으로 나타나는 패턴**이 관찰됩니다. 동거 자녀가 많은 군집에서 와인·육류 등의 지출이 상대적으로 적은지 추가로 비교해 볼 수 있습니다. 다만 군집별 평균만으로 동거 자녀 수가 지출 감소의 원인이라고 단정하기는 어려우므로, 연령과 소득 같은 다른 변수도 함께 확인하는 것이 좋습니다.


## 인사이트 정리 (예시)

- Elbow 그래프에서는 뚜렷한 꺾임점이 나타나지 않았고, Silhouette Score는 K=2에서 가장 높았습니다. 다만 K=2는 마케팅 활용에 필요한 세분화가 충분하지 않을 수 있어, 해석 가능성과 활용 목적을 함께 고려하여 K=4를 선택했습니다.
- 네 개의 군집은 소득·지출·동거 자녀 수의 조합에 따라 VIP, 우량, 자녀 동거 알뜰, 젊은 저관여 고객으로 구분할 수 있습니다.
- 소득 수준이 비슷한 고객도 동거 자녀 수에 따라 지출 패턴이 다르게 나타날 수 있습니다. 여러 변수를 함께 살펴보는 군집분석은 이러한 조합을 발견하는 데 도움이 됩니다.

과제인 이커머스 고객 세분화에서는 실제 거래 데이터로 RFM(Recency·Frequency·Monetary)을 계산한 뒤, 이번 실습과 같은 흐름으로 군집을 나누고 그 특징을 해석합니다.
